FashionMNIST 기반 VGG19 파인튜닝 전체 코드

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import torchvision.models as models



In [3]:


# 0. 디바이스 설정 (GPU 사용 가능 시 GPU 활용)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

########################################################
# 1. FashionMNIST 데이터셋 전처리 및 로드
########################################################
# VGG19의 입력 조건(3채널, 224x224)과 ImageNet 정규화 방식을 적용합니다.
transform = transforms.Compose([
    transforms.Resize((224, 224)),                       # 28x28 -> 224x224 리사이징
    transforms.Grayscale(num_output_channels=3),        # 1채널(흑백)을 3채널(RGB 형태)로 변환
    transforms.ToTensor(),                              # 텐서 변환 및 0~1 정규화
    transforms.Normalize(                               # ImageNet 사전 학습 모델 표준 정규화 통계치 적용
        mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225]
    )
])

# FashionMNIST 데이터셋 다운로드
full_train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

# 전체 학습 데이터(60,000개)를 학습용(55,000개)과 검증용(5,000개)으로 분할
train_size = 55000
val_size = 5000
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

# 데이터로더(DataLoader) 설정 (배치 크기 64)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)



Using device: cpu


In [10]:


########################################################
# 2. 사전 학습된 VGG19 모델 로드 및 구조 수정
########################################################
# ImageNet으로 사전 학습된(Pre-trained) VGG19 가중치를 불러옵니다.
model = models.vgg19(weights=models.VGG19_Weights.DEFAULT)
print(f'model: {dir(model)}')
print(f'classifier: {model.classifier}')

# VGG19는 classifier 모듈의 마지막 레이어(6번)가 최종 Fully Connected Layer입니다.
# classifier[6]의 입력 차원을 확인한 후 FashionMNIST의 클래스 수(10개)에 맞게 교체합니다.
num_features = model.classifier[6].in_features
model.classifier[6] = nn.Linear(num_features, 10)

print(f'classifier: {model.classifier}')

# 모델을 지정한 디바이스(GPU 또는 CPU)로 이동
model = model.to(device)



model: ['T_destination', '__annotations__', '__call__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_apply', '_backward_hooks', '_backward_pre_hooks', '_buffers', '_call_impl', '_compiled_call_impl', '_forward_hooks', '_forward_hooks_always_called', '_forward_hooks_with_kwargs', '_forward_pre_hooks', '_forward_pre_hooks_with_kwargs', '_get_backward_hooks', '_get_backward_pre_hooks', '_get_name', '_is_full_backward_hook', '_load_from_state_dict', '_load_state_dict_post_hooks', '_load_state_dict_pre_hooks', '_maybe_warn_non_full_backward_hook', '_modules', '_named_members', '_non_persistent_buffers_set', '_parameters', '_register_load_state_dict_

In [11]:


########################################################
# 3. 손실 함수와 옵티마이저 설정
########################################################
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)  # 파인튜닝을 위한 학습률 설정

########################################################
# 4. 학습 및 검증 루프 (Training & Validation Loop)
########################################################
epochs = 5  # 전이 학습 기준 설정 에폭
for epoch in range(epochs):
    # --- [Training] ---
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()               # 이전 기울기 초기화
        outputs = model(inputs)             # 순전파 (Forward pass)
        loss = criterion(outputs, labels)   # 손실 계산
        loss.backward()                     # 역전파 (Backward pass)
        optimizer.step()                    # 가중치 업데이트
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        train_total += labels.size(0)
        train_correct += predicted.eq(labels).sum().item()
        
    train_epoch_loss = train_loss / train_total
    train_epoch_acc = train_correct / train_total

    # --- [Validation] ---
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad(): # 검증 단계에서는 기울기 계산 비활성화
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()
            
    val_epoch_loss = val_loss / val_total
    val_epoch_acc = val_correct / val_total
    
    print(f"Epoch [{epoch+1}/{epochs}] | "
          f"Train Loss: {train_epoch_loss:.4f}, Train Acc: {train_epoch_acc*100:.2f}% | "
          f"Val Loss: {val_epoch_loss:.4f}, Val Acc: {val_epoch_acc*100:.2f}%")

########################################################
# 5. 테스트 평가 (Test Evaluation)
########################################################
model.eval()
test_loss, test_correct, test_total = 0.0, 0, 0
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        test_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        test_total += labels.size(0)
        test_correct += predicted.eq(labels).sum().item()

test_epoch_loss = test_loss / test_total
test_epoch_acc = test_correct / test_total
print(f"\n[Test Result] Loss: {test_epoch_loss:.4f}, Accuracy: {test_epoch_acc*100:.2f}%")

KeyboardInterrupt: 